# LLM-as-Judge
I'm using three foundation models to judge the dialogs generated my fine-tuned qwen model and my baseline model. The goal is to see which model is able to generate dialogs that convey the respective personalites better. Since I can't measure personality directly, I'm using classification approaches as a measurement proxy for the accuracy of personality display. On top of the traditional text classification approach, here, I am using LLM-as-Judge to also include reasoning capabilities employing qualitative coding methods.

In [1]:
from google import genai
from google.genai import types
import openai
from anthropic import Anthropic

import pandas as pd
import json
from json_repair import repair_json
import os
from dotenv import load_dotenv
import re

import krippendorff
import numpy as np

In [2]:
load_dotenv()

True

# Systemprompt

In [3]:
SYSTEMPROMPT = """
### ROLE
You are an expert psychometrician and qualitative researcher. Your task is to perform a Deductive Content Analysis derived from the method according to Philipp Mayring on the provided dialogues to identify MBTI personality preferences.

### METHODOLOGICAL RULES (The Coding Guide)
You must code the text based on the four dichotomies. An "Analytic Unit" is a single coherent statement within a turn of phrase.  If a segment does not contain any indicators for the four dichotomies, do not code it. The following keywords define the different categories.

1. CATEGORY: Extraversion (E) vs. Introversion (I)
   - Keywords (E): open, expressive, action-oriented, gregarious, active, enthusiastic
   - Keywords (I): private, quiet, contemplative, intimate, reflective, contained
2. CATEGORY: Sensing (S) vs. Intuition (N)
   - Keywords (S): concrete, realistic, present, practical, experiential, traditional
   - Keywords (N): abstract, imaginative, future, conceptual, theoretical, original
3. CATEGORY: Thinking (T) vs. Feeling (F)
	- Keywords (T): logical, reasonable, questioning, objective, critical, tough-minded
	- Keywords (F):empathetic, compassionate, accommodating, subjective, accepting, tender-hearted
4. CATEGORY: Judging (J) vs. Perceiving (P)
	- Keywords (J): systematic, planful, early starting, closure, scheduled, methodical
	- Keywords (P): casual, open-ended, pressure-prompted, options, spontaneous, emergent

### CODING PROCESS
For each dialogue, follow these three steps:
1. EXTRACTION: Identify specific text segments that serve as indicators for a dichotomy.
2. DEDUCTIVE ASSIGNMENT: Assign the category (e.g., "E") and provide a "Coding Rule" justification (Why does this segment fit the definition?). If a segment does not contain any indicators for the four dichotomies, do not code it.
3. SYNTHESIS: At the end of the dialogue, aggregate the findings to determine the most likely 4-letter type for Person A and Person B. You are NOT required to return a full label. However, you ARE REQUIRED to give a best guess of a full label. Give a short and concise reasoning for your choices, this should not be longer than one or two sentences.
### DIALOG FORMAT
It will be always two people talking with each other. The dialogues will have the following form:

[{'Szenario': 'Scenario Name',
  'Run': run_id,
  'content': Person A: "The weather is great! We should go water skiing or find some people to play beach volleyball!"\nPerson B: "I don't know about that. I would prefer to read my book in the park."\nPerson A: [...]},
  {'Scenario': ...
  {'Run': run_id,
  'content': ...},
  ...
]

### OUTPUT FORMAT (JSON-style preferred for API)
Please provide the analysis in this JSON structure:

{
    "dialogues": [
        {
            "global_id": 1,
            "codings": [
				{
					"text_segment": "We [...] should find some people to play beach volleyball!",
					"speaker": "Person A",
					"category": "Extraversion",
					"indicator": "E",
					"reasoning": "Matches the definition of seeking external social action and being active (beach volleyball).",
				  },
				  {
					"text_segment": "I would prefer to read my book in the park",
					"speaker": "Person B",
					"category": "Introversion",
					"indicator": "I",
					"reasoning": "Matches the definition of preferring private and quiet social situations (reading a book in the park)",
				},
				{
					"text_segment": "...",
					"...": "...",
				},
			],
			"final_conclusion":{
				"person_A": {
					"type": "Exxx", 
                    "type (best guess)": "ENTP",
					"summary": "Exhibits strong extraverted energy and logical flexibility..."
				},
				"person_B": {
					"type": "Ixxx", 
                    "type (best guess)": "INFP",
					"summary": "Shows a clear preference for quiet environments and internal reflection..."
				}
			},
		},
		{
            "global_id": 2,
			"codings": [
				...
			],
			"final_conclusion": {
				...
			},
			...
		}
	]
}

Respond ONLY with a valid JSON object. No markdown, no backticks, no explanation.
    

"""

# Dialogue Preparation

In [14]:
# preparing data
def prepare_data_for_api(raw_data):
    clean_data = []
    for entry in raw_data:
        # copy of data without MBTI types (since thats what we want to predict)
        dialogue_bundle = {
            "global_id": entry.get("global_id"), 
            "Szenario": entry.get("Szenario"),
            "Run": entry.get("Run"),
            "content": ""
        } # making a dict for every entry in raw data
        
        # combining utterances of speakers A and B into dialogue format 
        utterances_a = entry.get("Utterances_A", [])
        utterances_b = entry.get("Utterances_B", [])
        
        combined_text = ""
        for a, b in zip(utterances_a, utterances_b): # zip works like a zipper: index i from A then from B, i+1 from A then from B, etc
            combined_text += f"Person A: {a}\nPerson B: {b}\n" # marking whats person A and whats B
            
        dialogue_bundle["content"] = combined_text
        clean_data.append(dialogue_bundle)
    return clean_data



In [15]:
#dialogues generated by tuned model
# opening dialgogue json files
with open("../data/json/dialogues_comp_run5.json", "r") as file:
    raw_dialogues = json.load(file)

# adding ids
for i, dialogue in enumerate(raw_dialogues, start=1):
    dialogue['global_id'] = i

# writing indexed dialogues
with open('../data/json/dialogues_comp_run5.json', 'w') as f:
    json.dump(raw_dialogues, f, indent=4, ensure_ascii=False)

# table
for d in raw_dialogues:
    print(f"{d['global_id']:>3}  {d['Szenario']:<35} Run {d['Run']}  {d['MBTI_A']} vs {d['MBTI_B']}")


#dialogues generated by base model
with open("../data/json/dialogues_comp_base_run2.json", "r", encoding='latin-1') as file:
    raw_dialogues_base = json.load(file)

for i, dialogue in enumerate(raw_dialogues_base, start=1):
    dialogue['global_id'] = i

with open("../data/json/dialogues_comp_base_run2.json", "w", encoding='utf-8') as f:
    json.dump(raw_dialogues_base, f, indent=4, ensure_ascii=False)

for d in raw_dialogues:
    print(f"{d['global_id']:>3}  {d['Szenario']:<35} Run {d['Run']}  {d['MBTI_A']} vs {d['MBTI_B']}")

for d in raw_dialogues_base:
    print(f"{d['global_id']:>3}  {d['Szenario']:<35} Run {d['Run']}  {d['MBTI_A']} vs {d['MBTI_B']}")
DIALOGUES =prepare_data_for_api(raw_dialogues)
DIALOGUES_base=prepare_data_for_api(raw_dialogues_base)

  1  Work Place - Low Urgency            Run 1  INTJ vs ENFP
  2  Work Place - Low Urgency            Run 2  INTJ vs ENFP
  3  Work Place - Low Urgency            Run 1  ISTJ vs ESFP
  4  Work Place - Low Urgency            Run 2  ISTJ vs ESFP
  5  Work Place - Low Urgency            Run 1  ESTJ vs INFP
  6  Work Place - Low Urgency            Run 2  ESTJ vs INFP
  7  Crisis/Emergency - Medium Urgency   Run 1  INTJ vs ENFP
  8  Crisis/Emergency - Medium Urgency   Run 2  INTJ vs ENFP
  9  Crisis/Emergency - Medium Urgency   Run 1  ISTJ vs ESFP
 10  Crisis/Emergency - Medium Urgency   Run 2  ISTJ vs ESFP
 11  Crisis/Emergency - Medium Urgency   Run 1  ESTJ vs INFP
 12  Crisis/Emergency - Medium Urgency   Run 2  ESTJ vs INFP
 13  Residential/Community - High Urgency Run 1  INTJ vs ENFP
 14  Residential/Community - High Urgency Run 2  INTJ vs ENFP
 15  Residential/Community - High Urgency Run 1  ISTJ vs ESFP
 16  Residential/Community - High Urgency Run 2  ISTJ vs ESFP
 17  Residential/Com

In [ ]:
# making sure everythings as it should
len(DIALOGUES)

### less dialogues per turn
first = DIALOGUES[0:6]
second = DIALOGUES[6:12]
third = DIALOGUES[12:18]

print(len(first))
print(len(second))
print(len(third))

first_base = DIALOGUES_base[0:6]
second_base = DIALOGUES_base[6:12]
third_base = DIALOGUES_base[12:18]

print(len(first))
print(len(second))
print(len(third))

6
6
6
6
6
6


In [ ]:
DIALOGUES

[{'global_id': 1,
  'Szenario': 'Work Place - Low Urgency',
  'Run': 1,
  'content': "Person A: I think I'm going to do it. But we need to make sure there's time for us both to talk afterwards.'\nPerson B: You know me well enough to realize when I'm being serious, don't you? It sounds like a great idea. We can take turns leading meetings after this one, maybe even every week. That way everyone gets a chance to speak up!  You'll see if they're really interested in hearing our ideas. If people aren't paying attention during my part then maybe we shouldn't let them run\nPerson A: You've got a point... Ok fine, you win. Just please come over here tomorrow morning before lunch so we can go over all the details again.'\nPerson B: I would love to help out more often actually. Maybe you could ask me some questions too sometime?'\nPerson A: That'd be cool...'\nPerson B: Great thanks! And hey, why don't you just send an email now asking me those things? Then no matter which of us ends up doing t

In [11]:
DIALOGUES_base

[{'global_id': 1,
  'Szenario': 'Work Place - Low Urgency',
  'Run': 1,
  'content': "Person A: Alright, I think we should consider my approach for leading the next meeting since the project needs clear strategic direction and efficient decision-making moving forward. My analysis and planning can help us stay on track more effectively. What do you think? Have you had a chance to review the latest data on our progress? It might influence our decisions on who leads the meeting.\nPerson B: That's a great point, and I appreciate you bringing up the need for clarity and efficiency. However, I believe my insights could complement yours perfectly. With the upcoming changes in the team dynamics and some fresh ideas from recent brainstorming sessions, I think I can bring a different angle to the table. Plus, having someone flexible and adaptable like me leading the discussion might help us navigate these shifts smoothly.\nPerson A: I see where you're coming from, but letâ\x80\x99s focus on leve

# Analysis

In [ ]:
# gpt
def gpt_analyze(SYSTEMPROMPT, DIALOGUES):
    client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    user_content = json.dumps(DIALOGUES, ensure_ascii=False) if isinstance(DIALOGUES, (dict, list)) else str(DIALOGUES)

    try:
        response = client.chat.completions.create(
            model="gpt-5.1",  
            messages=[
                {"role": "system", "content": SYSTEMPROMPT},
                {"role": "user", "content": user_content}
            ],
            # json format
            response_format={"type": "json_object"},
            temperature=0,  ## might be conservative but better reproducability
            max_completion_tokens=16000 # context window by try and error, different for every model
        )
        
        # translating into dict
        result_json = json.loads(response.choices[0].message.content)
        return result_json

    except Exception as e:
        print(f"Fehler beim GPT-Aufruf: {e}")
        return None


# sonnet
def claude_analyze(SYSTEMPROMPT, DIALOGUES):
    client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
    user_content = json.dumps(DIALOGUES, ensure_ascii=False) if isinstance(DIALOGUES, (dict, list)) else str(DIALOGUES)

    try:
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=16000,
            temperature=0,
            system=SYSTEMPROMPT,
            messages=[
                {"role": "user", "content": user_content}
            ]
        )
        
        raw = response.content[0].text
        print(f"Stop reason: {response.stop_reason}")
        print(f"Länge raw: {len(raw)}")
        print(repr(raw[:50]))

        if "```" in raw:
            raw = raw.split("```json")[-1].split("```")[0].strip()

        result_json = json.loads(raw)

        #result_json = json.loads(response.content[0].text)
        return result_json  
    
    except Exception as e:
        print(f"Fehler beim Claude-Aufruf: {e}")
        return None

# gemini
def gemini_analyze(SYSTEMPROMPT, DIALOGUES):
    client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
    
    user_content = json.dumps(DIALOGUES, ensure_ascii=False) if isinstance(DIALOGUES, (dict, list)) else str(DIALOGUES)
    
    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=user_content,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEMPROMPT,
                response_mime_type="application/json",
                temperature=0,
                max_output_tokens=60000
            )
        )

        print(response.text)
        
        result_json = json.loads(response.text)
        return result_json
    
    except Exception as e:
        print(f"Fehler beim Gemini-Aufruf: {e}")
        return None

In [15]:
gpt_analysis_1 = gpt_analyze(SYSTEMPROMPT=SYSTEMPROMPT, DIALOGUES=first)
gpt_analysis_2 = gpt_analyze(SYSTEMPROMPT=SYSTEMPROMPT, DIALOGUES=second)
gpt_analysis_3 = gpt_analyze(SYSTEMPROMPT=SYSTEMPROMPT, DIALOGUES=third)

gpt_analysis_1_base = gpt_analyze(SYSTEMPROMPT=SYSTEMPROMPT, DIALOGUES=first_base)
gpt_analysis_2_base = gpt_analyze(SYSTEMPROMPT=SYSTEMPROMPT, DIALOGUES=second_base)
gpt_analysis_3_base = gpt_analyze(SYSTEMPROMPT=SYSTEMPROMPT, DIALOGUES=third_base)

In [16]:
gpt_analysis_1

{'dialogues': [{'global_id': 1,
   'codings': [{'text_segment': "we need to make sure there's time for us both to talk afterwards.",
     'speaker': 'Person A',
     'category': 'Judging',
     'indicator': 'J',
     'reasoning': 'Shows planful structuring of time and desire to schedule follow-up discussion.'},
    {'text_segment': 'We can take turns leading meetings after this one, maybe even every week. That way everyone gets a chance to speak up!',
     'speaker': 'Person B',
     'category': 'Extraversion',
     'indicator': 'E',
     'reasoning': 'Focuses on frequent group interaction and giving everyone a chance to speak, indicating social, outward orientation.'},
    {'text_segment': 'Then no matter which of us ends up doing the presentation nobody has an excuse for forgetting anything important!',
     'speaker': 'Person B',
     'category': 'Judging',
     'indicator': 'J',
     'reasoning': 'Wants to prevent forgetting by putting structure in place ahead of time, aiming for c

In [17]:
claude_analysis_1 = claude_analyze(SYSTEMPROMPT=SYSTEMPROMPT, DIALOGUES=first)
claude_analysis_2 = claude_analyze(SYSTEMPROMPT=SYSTEMPROMPT, DIALOGUES=second)
claude_analysis_3 = claude_analyze(SYSTEMPROMPT=SYSTEMPROMPT, DIALOGUES=third)

claude_analysis_1_base = claude_analyze(SYSTEMPROMPT=SYSTEMPROMPT, DIALOGUES=first_base)
claude_analysis_2_base = claude_analyze(SYSTEMPROMPT=SYSTEMPROMPT, DIALOGUES=second_base)
claude_analysis_3_base = claude_analyze(SYSTEMPROMPT=SYSTEMPROMPT, DIALOGUES=third_base)

Stop reason: end_turn
Länge raw: 27878
'```json\n{\n    "dialogues": [\n        {\n           '
Stop reason: end_turn
Länge raw: 32329
'```json\n{\n    "dialogues": [\n        {\n           '
Stop reason: end_turn
Länge raw: 45844
'```json\n{\n    "dialogues": [\n        {\n           '
Stop reason: end_turn
Länge raw: 38780
'```json\n{\n    "dialogues": [\n        {\n           '
Stop reason: end_turn
Länge raw: 48929
'```json\n{\n    "dialogues": [\n        {\n           '
Stop reason: end_turn
Länge raw: 26761
'```json\n{\n  "dialogues": [\n    {\n      "global_id"'


In [ ]:
claude_analysis_10

{'dialogues': [{'global_id': 1,
   'codings': [{'text_segment': "we need to make sure there's time for us both to talk afterwards",
     'speaker': 'Person A',
     'category': 'Judging',
     'indicator': 'J',
     'reasoning': 'Person A is planful and wants to schedule time for discussion, indicating a preference for structure and closure.'},
    {'text_segment': 'We can take turns leading meetings after this one, maybe even every week. That way everyone gets a chance to speak up!',
     'speaker': 'Person B',
     'category': 'Extraversion',
     'indicator': 'E',
     'reasoning': 'Person B is enthusiastic and action-oriented, proposing a structured social activity that involves everyone speaking up.'},
    {'text_segment': 'Just please come over here tomorrow morning before lunch so we can go over all the details again.',
     'speaker': 'Person A',
     'category': 'Judging',
     'indicator': 'J',
     'reasoning': 'Person A is scheduling a specific time and wants to go over det

In [19]:
gemini_analysis_1 = gemini_analyze(SYSTEMPROMPT=SYSTEMPROMPT, DIALOGUES=first)
gemini_analysis_2 = gemini_analyze(SYSTEMPROMPT=SYSTEMPROMPT, DIALOGUES=second)
gemini_analysis_3 = gemini_analyze(SYSTEMPROMPT=SYSTEMPROMPT, DIALOGUES=third)

gemini_analysis_1_base = gemini_analyze(SYSTEMPROMPT=SYSTEMPROMPT, DIALOGUES=first_base)
gemini_analysis_2_base = gemini_analyze(SYSTEMPROMPT=SYSTEMPROMPT, DIALOGUES=second_base)
gemini_analysis_3_base = gemini_analyze(SYSTEMPROMPT=SYSTEMPROMPT, DIALOGUES=third_base)

{
  "dialogues": [
    {
      "global_id": 1,
      "codings": [
        {
          "text_segment": "we need to make sure there's time for us both to talk afterwards.",
          "speaker": "Person A",
          "category": "Judging",
          "indicator": "J",
          "reasoning": "Matches the definition of being planful and systematic by ensuring time for a specific activity."
        },
        {
          "text_segment": "We can take turns leading meetings after this one, maybe even every week.",
          "speaker": "Person B",
          "category": "Judging",
          "indicator": "J",
          "reasoning": "Matches the definition of being systematic and scheduled by proposing a regular, structured rotation."
        },
        {
          "text_segment": "That way everyone gets a chance to speak up!",
          "speaker": "Person B",
          "category": "Feeling",
          "indicator": "F",
          "reasoning": "Matches the definition of being accommodating and accep

In [20]:
from beepmeup import beep
beep()

In [ ]:
#gemini_analysis


In [ ]:
# writing analysis into files
json_str = json.dumps(gpt_analysis_1, indent=4)
with open("..\data\json\gpt_analysis_1.json", "w") as f:
    f.write(json_str)

json_str = json.dumps(gpt_analysis_2, indent=4)
with open("..\data\json\gpt_analysis_2.json", "w") as f:
    f.write(json_str)

json_str = json.dumps(gpt_analysis_3, indent=4)
with open("..\data\json\gpt_analysis_3.json", "w") as f:
    f.write(json_str)

json_str = json.dumps(gpt_analysis_1_base, indent=4)
with open("..\data\json\gpt_analysis_base_1.json", "w") as f:
    f.write(json_str)

json_str = json.dumps(gpt_analysis_2_base, indent=4)
with open("..\data\json\gpt_analysis_base_2.json", "w") as f:
    f.write(json_str)

json_str = json.dumps(gpt_analysis_3_base, indent=4)
with open("..\data\json\gpt_analysis_base_3.json", "w") as f:
    f.write(json_str)

<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:6: SyntaxWarning: invalid escape sequence '\d'
<>:10: SyntaxWarning: invalid escape sequence '\d'
<>:14: SyntaxWarning: invalid escape sequence '\d'
<>:18: SyntaxWarning: invalid escape sequence '\d'
<>:22: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:6: SyntaxWarning: invalid escape sequence '\d'
<>:10: SyntaxWarning: invalid escape sequence '\d'
<>:14: SyntaxWarning: invalid escape sequence '\d'
<>:18: SyntaxWarning: invalid escape sequence '\d'
<>:22: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Tim\AppData\Local\Temp\ipykernel_24972\2991139184.py:2: SyntaxWarning: invalid escape sequence '\d'
  with open("..\data\json\gpt_analysis_1.json", "w") as f:
C:\Users\Tim\AppData\Local\Temp\ipykernel_24972\2991139184.py:6: SyntaxWarning: invalid escape sequence '\d'
  with open("..\data\json\gpt_analysis_2.json", "w") as f:
C:\Users\Tim\AppData\Local\Temp\ipykernel_24972\29911391

In [22]:
json_str = json.dumps(claude_analysis_1, indent=4)
with open("..\data\json\claude_analysis_1.json", "w") as f:
    f.write(json_str)

json_str = json.dumps(claude_analysis_2, indent=4)
with open("..\data\json\claude_analysis_2.json", "w") as f:
    f.write(json_str)

json_str = json.dumps(claude_analysis_3, indent=4)
with open("..\data\json\claude_analysis_3.json", "w") as f:
    f.write(json_str)


json_str = json.dumps(claude_analysis_1_base, indent=4)
with open("..\data\json\claude_analysis_base_1.json", "w") as f:
    f.write(json_str)

json_str = json.dumps(claude_analysis_2_base, indent=4)
with open("..\data\json\claude_analysis_base_2.json", "w") as f:
    f.write(json_str)

json_str = json.dumps(claude_analysis_3_base, indent=4)
with open("..\data\json\claude_analysis_base_3.json", "w") as f:
    f.write(json_str)

<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:6: SyntaxWarning: invalid escape sequence '\d'
<>:10: SyntaxWarning: invalid escape sequence '\d'
<>:15: SyntaxWarning: invalid escape sequence '\d'
<>:19: SyntaxWarning: invalid escape sequence '\d'
<>:23: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:6: SyntaxWarning: invalid escape sequence '\d'
<>:10: SyntaxWarning: invalid escape sequence '\d'
<>:15: SyntaxWarning: invalid escape sequence '\d'
<>:19: SyntaxWarning: invalid escape sequence '\d'
<>:23: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Tim\AppData\Local\Temp\ipykernel_24972\4191275463.py:2: SyntaxWarning: invalid escape sequence '\d'
  with open("..\data\json\claude_analysis_1.json", "w") as f:
C:\Users\Tim\AppData\Local\Temp\ipykernel_24972\4191275463.py:6: SyntaxWarning: invalid escape sequence '\d'
  with open("..\data\json\claude_analysis_2.json", "w") as f:
C:\Users\Tim\AppData\Local\Temp\ipykernel_24972\41

In [23]:
json_str = json.dumps(gemini_analysis_1, indent=4)
with open("..\data\json\gemini_analysis_1.json", "w") as f:
    f.write(json_str)

json_str = json.dumps(gemini_analysis_2, indent=4)
with open("..\data\json\gemini_analysis_2.json", "w") as f:
    f.write(json_str)

json_str = json.dumps(gemini_analysis_3, indent=4)
with open("..\data\json\gemini_analysis_3.json", "w") as f:
    f.write(json_str)

json_str = json.dumps(gemini_analysis_1_base, indent=4)
with open("..\data\json\gemini_analysis_base_1.json", "w") as f:
    f.write(json_str)

json_str = json.dumps(gemini_analysis_2_base, indent=4)
with open("..\data\json\gemini_analysis_base_2.json", "w") as f:
    f.write(json_str)

json_str = json.dumps(gemini_analysis_3_base, indent=4)
with open("..\data\json\gemini_analysis_base_3.json", "w") as f:
    f.write(json_str)

<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:6: SyntaxWarning: invalid escape sequence '\d'
<>:10: SyntaxWarning: invalid escape sequence '\d'
<>:14: SyntaxWarning: invalid escape sequence '\d'
<>:18: SyntaxWarning: invalid escape sequence '\d'
<>:22: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:6: SyntaxWarning: invalid escape sequence '\d'
<>:10: SyntaxWarning: invalid escape sequence '\d'
<>:14: SyntaxWarning: invalid escape sequence '\d'
<>:18: SyntaxWarning: invalid escape sequence '\d'
<>:22: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Tim\AppData\Local\Temp\ipykernel_24972\3768650688.py:2: SyntaxWarning: invalid escape sequence '\d'
  with open("..\data\json\gemini_analysis_1.json", "w") as f:
C:\Users\Tim\AppData\Local\Temp\ipykernel_24972\3768650688.py:6: SyntaxWarning: invalid escape sequence '\d'
  with open("..\data\json\gemini_analysis_2.json", "w") as f:
C:\Users\Tim\AppData\Local\Temp\ipykernel_24972\37

In [3]:
from pathlib import Path

# reading rater analysis files into dict
def merge_rater_files(file_list):
    merged = {}
    for path in file_list: # opening files
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        for category, dialogues in data.items(): #
            if category not in merged: # if category is not in new file, create it
                merged[category] = []
            merged[category].extend(dialogues) # extend instead of append to not have nested list
    return merged

# Alle Dateien eines Raters zusammenführen
gpt = merge_rater_files([
    '../data/json/gpt_analysis_1.json',
    '../data/json/gpt_analysis_2.json',
    '../data/json/gpt_analysis_3.json',
])

opus = merge_rater_files([
    '../data/json/claude_analysis_1.json',
    '../data/json/claude_analysis_2.json',
    '../data/json/claude_analysis_3.json',
])

gemini = merge_rater_files([
    '../data/json/gemini_analysis_1.json',
    '../data/json/gemini_analysis_2.json',
    '../data/json/gemini_analysis_3.json',
])


# not sure why I did different ways here, probably to try it out
def merge_rater_glob(pattern):
    files = sorted(Path('.').glob(pattern))
    return merge_rater_files(files)

#gpt    = merge_rater_glob('../data/json/gpt_analysis_*.json')
# opus   = merge_rater_glob('../data/json/claude_analysis_*.json')
# gemini = merge_rater_glob('../data/json/gemini_analysis_*.json')

gpt_base    = merge_rater_glob('../data/json/gpt_analysis_base_*.json')
opus_base   = merge_rater_glob('../data/json/claude_analysis_base_*.json')
gemini_base = merge_rater_glob('../data/json/gemini_analysis_base_*.json')

In [ ]:
# gpt_files    = sorted(Path('../data/json').glob('gpt_analysis_*.json'))
# opus_files   = sorted(Path('../data/json').glob('claude_analysis_*.json'))
# gemini_files = sorted(Path('../data/json').glob('gemini_analysis_*.json'))

# print("GPT:", gpt_files)
# print("Opus:", opus_files)
# print("Gemini:", gemini_files)

GPT: [WindowsPath('../data/json/gpt_analysis_1.json'), WindowsPath('../data/json/gpt_analysis_2.json'), WindowsPath('../data/json/gpt_analysis_3.json')]
Opus: [WindowsPath('../data/json/claude_analysis_1.json'), WindowsPath('../data/json/claude_analysis_2.json'), WindowsPath('../data/json/claude_analysis_3.json')]
Gemini: [WindowsPath('../data/json/gemini_analysis_1.json'), WindowsPath('../data/json/gemini_analysis_2.json'), WindowsPath('../data/json/gemini_analysis_3.json')]


In [4]:
first_key = list(gpt.keys())[0]
print(type(gpt[first_key]))        # List or dict? --> list
print(gpt[first_key][0] if isinstance(gpt[first_key], list) else list(gpt[first_key].keys())[:5]) # probing json structure, if first key is a list, give content, if not give first 5 keys

# translating json into flat table (dict): global_id | speaker | rater | type | type_guess
def extract_labels(data, rater_name):
    rows = []
    for dialogue in data['dialogues']: # for every dialogue
        did = dialogue['global_id'] # safe id 
        for speaker, vals in dialogue['final_conclusion'].items(): # for every speaker (A & B) put keys into row of rows. 
            # --> for speaker, vals in blabla.item() does this: ( "Speaker_A" , {"type": "E", "type (best guess)": "ENFP"} ) Speaker_A is speaker
            # vals is {"type": "E", "type (best guess)": "ENFP"}, becomes tuple: ("Speaker_A", {"type": "E", "type (best guess)": "ENFP"})
            # --> so I can use e.g. vals["type"] to fetch type etc
            rows.append({
                'global_id': did,
                'speaker': speaker,
                'rater': rater_name,
                'type_evidence': vals['type'],
                'type_guess': vals['type (best guess)']
            })
    return pd.DataFrame(rows)

df = pd.concat([
    extract_labels(gpt,    'GPT'),
    extract_labels(opus,   'Opus'),
    extract_labels(gemini, 'Gemini')
])

df_base = pd.concat([
    extract_labels(gpt_base,    'GPT'),
    extract_labels(opus_base,   'Opus'),
    extract_labels(gemini_base, 'Gemini')
])

df['unit'] = df['global_id'].astype(str) + '_' + df['speaker']

df_base['unit'] = df_base['global_id'].astype(str) + '_' + df_base['speaker']

<class 'list'>
{'global_id': 1, 'codings': [{'text_segment': "we need to make sure there's time for us both to talk afterwards.", 'speaker': 'Person A', 'category': 'Judging', 'indicator': 'J', 'reasoning': 'Shows planful structuring of time and desire to schedule follow-up discussion.'}, {'text_segment': 'We can take turns leading meetings after this one, maybe even every week. That way everyone gets a chance to speak up!', 'speaker': 'Person B', 'category': 'Extraversion', 'indicator': 'E', 'reasoning': 'Focuses on frequent group interaction and giving everyone a chance to speak, indicating social, outward orientation.'}, {'text_segment': 'Then no matter which of us ends up doing the presentation nobody has an excuse for forgetting anything important!', 'speaker': 'Person B', 'category': 'Judging', 'indicator': 'J', 'reasoning': 'Wants to prevent forgetting by putting structure in place ahead of time, aiming for closure and preparedness.'}, {'text_segment': "wouldn't it be easier to 

In [ ]:
# Mapping labelsn
# For type_guess complete 0,1,2,3
# Für type_evidence not complete, x for missing

def extract_dim(label, dim):
    """Extracts I/E, N/S, F/T, P/J; NaN if 'x' or missing."""
    dim_map = {'IE': 0, 'NS': 1, 'FT': 2, 'PJ': 3}
    pairs   = {'IE': ('I','E'), 'NS': ('N','S'), 'FT': ('F','T'), 'PJ': ('P','J')}
    
    if not isinstance(label, str):
        return np.nan
    
    label = label.upper()
    valid = pairs[dim]
    
    # search letters (dimensions) in labels
    for ch in label:
        if ch in valid:
            # position correct?
            if len(label) == 4:
                return ch if label[dim_map[dim]] == ch else np.nan
            return ch  # if incomplete just take it
    return np.nan #if nothing return nan

dimensionen = ['IE', 'NS', 'FT', 'PJ']


# creating new columns evidence_dim / guess_dim
for dim in dimensionen:
    df[f'evidence_{dim}'] = df['type_evidence'].apply(lambda x: extract_dim(x, dim))
    df[f'guess_{dim}']    = df['type_guess'].apply(lambda x: extract_dim(x, dim))


for dim in dimensionen:
    df_base[f'evidence_{dim}'] = df_base['type_evidence'].apply(lambda x: extract_dim(x, dim))
    df_base[f'guess_{dim}']    = df_base['type_guess'].apply(lambda x: extract_dim(x, dim))


# Krippendorffs Alpha

In [ ]:
def compute_alpha(df, col, dim):
    col_name = f'{col}_{dim}'
    
    # pivot table
    # basically shows labels per text per rater per speaker
    pivot = df.pivot_table(
        index='rater',
        columns='unit',
        values=col_name,
        aggfunc='first'
    )
    
    #print(pivot)
#
    pairs = {'IE': {'I': 0, 'E': 1}, 'NS': {'N': 0, 'S': 1},
             'FT': {'F': 0, 'T': 1}, 'PJ': {'P': 0, 'J': 1}}
    matrix = pivot.replace(pairs[dim]).astype(float).values # translating into numeric
    
    #print(matrix)

    unique_vals = pd.unique(matrix[~np.isnan(matrix)])
    if len(unique_vals) <= 1:
        return np.nan
    
    return krippendorff.alpha(matrix, level_of_measurement='nominal')

# table for fine-tuned model
print("Fine-tuned: ")
print(f"{'Dim':<6} {'Evidence α':>12} {'Guess α':>12}")
print("-" * 32)
for dim in dimensionen:
    a_ev = compute_alpha(df, 'evidence', dim)
    a_gu = compute_alpha(df, 'guess',    dim)
    ev_str = f"{a_ev:>12.4f}" if not np.isnan(a_ev) else f"{'no variance':>12}"
    gu_str = f"{a_gu:>12.4f}" if not np.isnan(a_gu) else f"{'no variance':>12}"
    print(f"{dim:<6} {ev_str} {gu_str}")

Fine-tuned
Dim      Evidence α      Guess α
--------------------------------
IE           0.5368       0.3977
NS           0.3397       0.1739
FT           0.3269       0.2989
PJ           0.4814       0.3947


In [ ]:
# table for base model
print("Base: ")
print(f"{'Dim':<6} {'Evidence α':>12} {'Guess α':>12}")
for dim in dimensionen:
    a_ev = compute_alpha(df_base, 'evidence', dim)
    a_gu = compute_alpha(df_base, 'guess',    dim)
    ev_str = f"{a_ev:>12.4f}" if not np.isnan(a_ev) else f"{'no variance':>12}"
    gu_str = f"{a_gu:>12.4f}" if not np.isnan(a_gu) else f"{'no variance':>12}"
    print(f"{dim:<6} {ev_str} {gu_str}")

Base
Dim      Evidence α      Guess α
IE           0.2387       0.3808
NS           0.8125       0.7906
FT           0.7060       0.7060
PJ           0.4513       0.4513


In [11]:
print(list(gpt.keys()))
print(list(opus.keys()))
print(list(gemini.keys()))

['dialogues']
['dialogues']
['dialogues']


In [16]:
print("GT IDs:", sorted(d['global_id'] for d in raw_dialogues))
print("GPT IDs:", sorted(d['global_id'] for d in gpt['dialogues']))
print("Opus IDs:", sorted(d['global_id'] for d in opus['dialogues']))
print("Gemini IDs:", sorted(d['global_id'] for d in gemini['dialogues']))

GT IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
GPT IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
Opus IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
Gemini IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]


In [17]:
# Was sind die zwei Einträge mit global_id 1?
for d in gpt['dialogues']:
    if d['global_id'] == 1:
        print(d['final_conclusion'])

{'person_A': {'type': 'xSxJ', 'type (best guess)': 'ESFJ', 'summary': 'Shows a structured, planful approach to meetings and follow-ups, with a social, cooperative tone; limited evidence on S/N and T/F but language leans practical and people-focused.'}, 'person_B': {'type': 'ExxJ', 'type (best guess)': 'ENFJ', 'summary': 'Appears socially oriented and group-focused, wanting everyone to speak and be heard, while also emphasizing advance organization and written details.'}}


In [18]:
def parse_gt(raw_data):
    rows = []
    for entry in raw_data:
        for speaker, mbti_key in [('person_A', 'MBTI_A'), ('person_B', 'MBTI_B')]:
            mbti = entry[mbti_key]
            rows.append({
                'unit': f"{entry['global_id']}_{speaker}",
                'IE': mbti[0],
                'NS': mbti[1],
                'FT': mbti[2],
                'PJ': mbti[3],
            })
    return pd.DataFrame(rows).set_index('unit')

In [ ]:
#not quite sure why I used extract_dim again, seems redundant
# for dim in dimensionen:
#     df[f'evidence_{dim}'] = df['type_evidence'].apply(lambda x: extract_dim(x, dim))
#     df[f'guess_{dim}']    = df['type_guess'].apply(lambda x: extract_dim(x, dim))

gt = parse_gt(raw_dialogues)

In [ ]:
# !this also does not make any sense, since gt is the same for both since the type pairs are the same for base and fine-tuned dialogues!


# for dim in dimensionen:
#     df_base[f'evidence_{dim}'] = df_base['type_evidence'].apply(lambda x: extract_dim(x, dim))
#     df_base[f'guess_{dim}']    = df_base['type_guess'].apply(lambda x: extract_dim(x, dim))

# gt = parse_gt(raw_dialogues_base)

In [ ]:
# Metrics for fine-tuned dialogues
# dimension basis
from sklearn.metrics import f1_score, accuracy_score

dimensionen = ['IE', 'NS', 'FT', 'PJ']

print(f"{'Rater':<8} {'Dim':<6} {'Acc (ev)':>10} {'F1 (ev)':>10} {'Acc (gu)':>10} {'F1 (gu)':>10}")
print("-" * 55)

for rater in ['GPT', 'Opus', 'Gemini']:
    rater_df = df[df['rater'] == rater].set_index('unit')
    
    for dim in dimensionen:
        results = []
        for col in ['evidence', 'guess']:
            col_name = f'{col}_{dim}'
            common = rater_df.index.intersection(gt.index)
            y_pred = rater_df.loc[common, col_name].dropna()
            y_true = gt.loc[y_pred.index, dim]
            mask = y_pred.notna() & y_true.notna()
            y_pred, y_true = y_pred[mask], y_true[mask]
            
            if len(y_pred) == 0:
                results.extend([np.nan, np.nan])
                continue
            
            acc = accuracy_score(y_true, y_pred)
            f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
            results.extend([acc, f1])
        
        print(f"{rater:<8} {dim:<6} {results[0]:>10.3f} {results[1]:>10.3f} {results[2]:>10.3f} {results[3]:>10.3f}")
    print()

Rater    Dim      Acc (ev)    F1 (ev)   Acc (gu)    F1 (gu)
-------------------------------------------------------
GPT      IE          0.600      0.504      0.556      0.500
GPT      NS          0.483      0.480      0.472      0.469
GPT      FT          0.581      0.579      0.583      0.575
GPT      PJ          0.556      0.518      0.556      0.518

Opus     IE          0.429      0.300      0.444      0.429
Opus     NS          0.647      0.575      0.528      0.496
Opus     FT          0.727      0.725      0.750      0.750
Opus     PJ          0.536      0.482      0.556      0.518

Gemini   IE          0.444      0.398      0.444      0.398
Gemini   NS          0.486      0.482      0.500      0.498
Gemini   FT          0.694      0.682      0.694      0.682
Gemini   PJ          0.639      0.639      0.639      0.639



In [ ]:
# Metrics for base dialogues
# dimension basis

dimensionen = ['IE', 'NS', 'FT', 'PJ']

print(f"{'Rater':<8} {'Dim':<6} {'Acc (ev)':>10} {'F1 (ev)':>10} {'Acc (gu)':>10} {'F1 (gu)':>10}")
print("-" * 55)

for rater in ['GPT', 'Opus', 'Gemini']:
    rater_df = df_base[df_base['rater'] == rater].set_index('unit')
    
    for dim in dimensionen:
        results = []
        for col in ['evidence', 'guess']:
            col_name = f'{col}_{dim}'
            common = rater_df.index.intersection(gt.index)
            y_pred = rater_df.loc[common, col_name].dropna()
            y_true = gt.loc[y_pred.index, dim]
            mask = y_pred.notna() & y_true.notna()
            y_pred, y_true = y_pred[mask], y_true[mask]
            
            if len(y_pred) == 0:
                results.extend([np.nan, np.nan])
                continue
            
            acc = accuracy_score(y_true, y_pred)
            f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
            results.extend([acc, f1])
        
        print(f"{rater:<8} {dim:<6} {results[0]:>10.3f} {results[1]:>10.3f} {results[2]:>10.3f} {results[3]:>10.3f}")
    print()

Rater    Dim      Acc (ev)    F1 (ev)   Acc (gu)    F1 (gu)
-------------------------------------------------------
GPT      IE          0.524      0.464      0.528      0.479
GPT      NS          0.750      0.748      0.667      0.662
GPT      FT          0.833      0.833      0.833      0.833
GPT      PJ          0.556      0.476      0.556      0.476

Opus     IE            nan        nan      0.611      0.610
Opus     NS          0.629      0.608      0.667      0.649
Opus     FT          0.917      0.916      0.917      0.916
Opus     PJ          0.833      0.829      0.833      0.829

Gemini   IE          0.639      0.632      0.639      0.632
Gemini   NS          0.639      0.625      0.639      0.625
Gemini   FT          0.917      0.916      0.917      0.916
Gemini   PJ          0.778      0.775      0.778      0.775



In [ ]:
# Metrics fine-tuned dialogues
# class basis

for model_name, model_df in [('Fine-tuned', df), ('Base', df_base)]:
    print(f"\n{model_name}")
    print(f"{'Rater':<8} {'Acc (ev)':>10} {'F1 (ev)':>10} {'Acc (gu)':>10} {'F1 (gu)':>10}")
    print("-" * 48)
    
    for rater in ['GPT', 'Opus', 'Gemini']:
        accs_ev, f1s_ev, accs_gu, f1s_gu = [], [], [], []
        rater_df = model_df[model_df['rater'] == rater].set_index('unit')
        
        for dim in dimensionen:
            for col, acc_list, f1_list in [('evidence', accs_ev, f1s_ev), 
                                            ('guess', accs_gu, f1s_gu)]:
                col_name = f'{col}_{dim}'
                common = rater_df.index.intersection(gt.index)
                y_pred = rater_df.loc[common, col_name].dropna()
                y_true = gt.loc[y_pred.index, dim]
                mask = y_pred.notna() & y_true.notna()
                y_pred, y_true = y_pred[mask], y_true[mask]
                
                if len(y_pred) == 0:
                    continue
                
                acc_list.append(accuracy_score(y_true, y_pred))
                f1_list.append(f1_score(y_true, y_pred, average='macro', zero_division=0))
        
        print(f"{rater:<8} {np.mean(accs_ev):>10.3f} {np.mean(f1s_ev):>10.3f} {np.mean(accs_gu):>10.3f} {np.mean(f1s_gu):>10.3f}")


Fine-tuned
Rater      Acc (ev)    F1 (ev)   Acc (gu)    F1 (gu)
------------------------------------------------
GPT           0.555      0.520      0.542      0.516
Opus          0.585      0.521      0.569      0.548
Gemini        0.566      0.550      0.569      0.554

Base
Rater      Acc (ev)    F1 (ev)   Acc (gu)    F1 (gu)
------------------------------------------------
GPT           0.666      0.630      0.646      0.613
Opus          0.793      0.784      0.757      0.751
Gemini        0.743      0.737      0.743      0.737


# Language Detection

Looking through some dialogues showed that the base model sprinkled some chinese language snippets, which needs investigation.

In [ ]:
# load data in utf-8, to test data quality, especially for chinese characters
with open('../data/json/test/dialogues_comp_base_run2.json', 'r', encoding='utf-8') as f:
    raw_dialogues_base = json.load(f)

In [88]:
with open("../data/json/test/dialogues_comp_base_run2.json", "r", encoding='utf-8') as file:
    raw_dialogues_base = json.load(file)

for i, dialogue in enumerate(raw_dialogues_base, start=1):
    dialogue['global_id'] = i

for d in raw_dialogues:
    print(f"{d['global_id']:>3}  {d['Szenario']:<35} Run {d['Run']}  {d['MBTI_A']} vs {d['MBTI_B']}")

for d in raw_dialogues_base:
    print(f"{d['global_id']:>3}  {d['Szenario']:<35} Run {d['Run']}  {d['MBTI_A']} vs {d['MBTI_B']}")
DIALOGUES =prepare_data_for_api(raw_dialogues)
DIALOGUES_base=prepare_data_for_api(raw_dialogues_base)

  1  Work Place - Low Urgency            Run 1  INTJ vs ENFP
  2  Work Place - Low Urgency            Run 2  INTJ vs ENFP
  3  Work Place - Low Urgency            Run 1  ISTJ vs ESFP
  4  Work Place - Low Urgency            Run 2  ISTJ vs ESFP
  5  Work Place - Low Urgency            Run 1  ESTJ vs INFP
  6  Work Place - Low Urgency            Run 2  ESTJ vs INFP
  7  Crisis/Emergency - Medium Urgency   Run 1  INTJ vs ENFP
  8  Crisis/Emergency - Medium Urgency   Run 2  INTJ vs ENFP
  9  Crisis/Emergency - Medium Urgency   Run 1  ISTJ vs ESFP
 10  Crisis/Emergency - Medium Urgency   Run 2  ISTJ vs ESFP
 11  Crisis/Emergency - Medium Urgency   Run 1  ESTJ vs INFP
 12  Crisis/Emergency - Medium Urgency   Run 2  ESTJ vs INFP
 13  Residential/Community - High Urgency Run 1  INTJ vs ENFP
 14  Residential/Community - High Urgency Run 2  INTJ vs ENFP
 15  Residential/Community - High Urgency Run 1  ISTJ vs ESFP
 16  Residential/Community - High Urgency Run 2  ISTJ vs ESFP
 17  Residential/Com

In [65]:
from langdetect import detect

def detect_language(text):
    try:
        return detect(text)
    except:
        return 'unknown'
    
def extract_utterances(raw_data, model_name):
    rows = []
    for dialogue in raw_data:
        for utt in dialogue['Utterances_A']:
            rows.append({'model': model_name, 'speaker': 'A', 
                        'global_id': dialogue['global_id'], 'text': utt})
        for utt in dialogue['Utterances_B']:
            rows.append({'model': model_name, 'speaker': 'B', 
                        'global_id': dialogue['global_id'], 'text': utt})
    return pd.DataFrame(rows)

df_utterances = pd.concat([
    extract_utterances(raw_dialogues,      'fine-tuned'),
    extract_utterances(raw_dialogues_base, 'base')
])

print(df_utterances.shape)
print(df_utterances.head())

(504, 4)
        model speaker  global_id  \
0  fine-tuned       A          1   
1  fine-tuned       A          1   
2  fine-tuned       A          1   
3  fine-tuned       A          1   
4  fine-tuned       A          1   

                                                text  
0  I think I'm going to do it. But we need to mak...  
1  You've got a point... Ok fine, you win. Just p...  
2                                 That'd be cool...'  
3  Ok, I guess I could do that... but wouldn't it...  
4  Well ok, since you asked nicely... but don't b...  


In [66]:
df_utterances['language'] = df_utterances['text'].apply(detect_language)

# Übersicht
print(df_utterances.groupby(['model', 'language']).size())

model       language
base        ca            1
            en          242
            fr            1
            so            1
            sw            1
            zh-cn         6
fine-tuned  af            1
            en          246
            so            1
            sw            1
            tl            1
            tr            2
dtype: int64


In [67]:
# Was sind die nicht-englischen Utterances?
mask = df_utterances['language'] != 'en'
print(df_utterances[mask][['model', 'language', 'text']].to_string())


          model language                                                                                                                                                                                                                                                                                                      text
12   fine-tuned       tl                                                                                                                                                                                                                                                                                      Bye! Happy planning!
40   fine-tuned       tr                                                                                                                                                                Ok. Bye! Bye-bye!!! Bye! Bye!!! Bye!!! Bye!!!! Bye!!!! Bye!!!!! Bye!!!!!!!! Bye!!!!!! Bye!!!!! Bye!!!! Bye!!! Bye!! Bye! Bye!! Bye!!! Bye!
117  fine-tuned       sw       

In [68]:
import unicodedata

def contains_cjk(text):
    """Prüft ob Text chinesische/japanische/koreanische Zeichen enthält."""
    return any(unicodedata.name(c, '').startswith('CJK') for c in text)

df_utterances['has_cjk'] = df_utterances['text'].apply(contains_cjk)

print(df_utterances.groupby('model')['has_cjk'].sum())
print(df_utterances.groupby('model')['has_cjk'].mean())  # Anteil

model
base          11
fine-tuned     0
Name: has_cjk, dtype: int64
model
base          0.043651
fine-tuned    0.000000
Name: has_cjk, dtype: float64


In [71]:
df_utterances.to_csv("../data/csv/df_utterances.csv", 
                     index=False,        # kein extra Index
                     encoding='utf-8',   # wegen der CJK-Zeichen
                     sep=';')

# Significance Tests

*Note*: Usnig Wilcoxon here was not correct, needs to be corrected to McNemar's Test

In [ ]:
# from scipy.stats import wilcoxon

# print(f"{'Rater':<8} {'Dim':<6} {'W':>8} {'p':>8}")
# print("-" * 35)

# for rater in ['GPT', 'Opus', 'Gemini']:
#     for dim in dimensionen:
#         # F1 pro Unit für beide Modelle
#         results = []
#         for model_df, gt_local in [(df, gt), (df_base, gt)]:
#             rater_df = model_df[model_df['rater'] == rater].set_index('unit')
#             common = rater_df.index.intersection(gt_local.index)
#             y_pred = rater_df.loc[common, f'guess_{dim}'].dropna()
#             y_true = gt_local.loc[y_pred.index, dim]
#             mask = y_pred.notna() & y_true.notna()
#             correct = (y_pred[mask] == y_true[mask]).astype(int)
#             results.append(correct)
        
#         # Gemeinsame Units
#         common_units = results[0].index.intersection(results[1].index)
#         if len(common_units) < 10:
#             print(f"{rater:<8} {dim:<6} {'too few samples':>18}")
#             continue
        
#         a = results[0].loc[common_units]
#         b = results[1].loc[common_units]
        
#         if (a == b).all():
#             print(f"{rater:<8} {dim:<6} {'no difference':>18}")
#             continue
        
#         stat, p = wilcoxon(a, b)
#         sig = '*' if p < 0.05 else ''
#         print(f"{rater:<8} {dim:<6} {stat:>8.3f} {p:>8.4f} {sig}")

Rater    Dim           W        p
-----------------------------------
GPT      IE       30.000   0.7630 
GPT      NS       60.000   0.1083 
GPT      FT       50.000   0.0389 *
GPT      PJ       18.000   1.0000 
Opus     IE       73.500   0.1797 
Opus     NS       18.000   0.1317 
Opus     FT       11.000   0.0578 
Opus     PJ       15.000   0.0075 *
Gemini   IE       45.000   0.0896 
Gemini   NS       70.000   0.2513 
Gemini   FT        5.500   0.0114 *
Gemini   PJ       54.000   0.2253 


In [ ]:
# from scipy.stats import wilcoxon, norm
# import pingouin as pg
# import numpy as np

# print(f"{'Rater':<8} {'Dim':<6} {'W':>8} {'p':>8} {'r':>8} {'Power':>8}")
# print("-" * 55)

# for rater in ['GPT', 'Opus', 'Gemini']:
#     for dim in dimensionen:
#         results = []
#         for model_df in [df, df_base]:
#             rater_df = model_df[model_df['rater'] == rater].set_index('unit')
#             common = rater_df.index.intersection(gt.index)
#             y_pred = rater_df.loc[common, f'guess_{dim}'].dropna()
#             y_true = gt.loc[y_pred.index, dim]
#             mask = y_pred.notna() & y_true.notna()
#             # Binary accuracy: 1 if correct, 0 if wrong
#             correct = (y_pred[mask] == y_true[mask]).astype(int)
#             results.append(correct)
        
#         common_units = results[0].index.intersection(results[1].index)
#         n = len(common_units)
        
#         # Check if we have enough data or if there's zero variance
#         if n < 10 or (results[0].loc[common_units] == results[1].loc[common_units]).all():
#             print(f"{rater:<8} {dim:<6} {'skipped':>25}")
#             continue
        
#         a = results[0].loc[common_units]
#         b = results[1].loc[common_units]
        
#         stat, p = wilcoxon(a, b)
        
#         # Effect size r = Z / sqrt(N)
#         # Using clip to avoid infinity if p is exactly 0 or 1
#         z = abs(norm.ppf(np.clip(p / 2, 1e-12, 0.5)))
#         r = z / np.sqrt(n)
        
#         # Post-hoc Power 
#         # Using pg.power_ttest as the standard approximation for Wilcoxon power
#         # contrast='paired' matches the Wilcoxon Signed-Rank design
#         power = pg.power_ttest(d=r, n=n, contrast='paired', alpha=0.05)
        
#         sig = '*' if p < 0.05 else ''

#         # Required N for power = 0.80
#         try:
#             required_n = pg.power_ttest(d=r, power=0.80, contrast='paired', alpha=0.05)
#             required_n = int(np.ceil(required_n))
#         except Exception:
#             required_n = None

#         required_str = str(required_n) if required_n else 'N/A'
#         print(f"{rater:<8} {dim:<6} {stat:>8.3f} {p:>8.4f} {r:>8.3f} {power:>8.3f} {required_str:>12} {sig}")
#         #print(f"{rater:<8} {dim:<6} {stat:>8.3f} {p:>8.4f} {r:>8.3f} {power:>8.3f} {sig}")

Rater    Dim           W        p        r    Power
-------------------------------------------------------
GPT      IE       30.000   0.7630    0.050    0.060         3111 
GPT      NS       60.000   0.1083    0.268    0.346          112 
GPT      FT       50.000   0.0389    0.344    0.519           69 *
GPT      PJ       18.000   1.0000    0.000    0.050          N/A 
Opus     IE       73.500   0.1797    0.224    0.257          159 
Opus     NS       18.000   0.1317    0.251    0.311          127 
Opus     FT       11.000   0.0578    0.316    0.454           81 
Opus     PJ       15.000   0.0075    0.445    0.738           42 *
Gemini   IE       45.000   0.0896    0.283    0.379          100 
Gemini   NS       70.000   0.2513    0.191    0.200          217 
Gemini   FT        5.500   0.0114    0.422    0.691           47 *
Gemini   PJ       54.000   0.2253    0.202    0.218          195 


## McNemar's Test

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
import numpy as np

# mcnemar looks at dicodant pairs (b und c)
# b = fine-tuned right, base wrong
# c = fine-tuned wrong, base right

print(f"{'Rater':<8} {'Dim':<6} {'b':>5} {'c':>5} {'p':>8} {'n_needed':>10}")
print("-" * 45)

for rater in ['GPT', 'Opus', 'Gemini']:
    for dim in dimensionen:
        results = []
        for model_df in [df, df_base]:
            rater_df = model_df[model_df['rater'] == rater].set_index('unit')
            common = rater_df.index.intersection(gt.index)
            y_pred = rater_df.loc[common, f'guess_{dim}'].dropna()
            y_true = gt.loc[y_pred.index, dim]
            mask = y_pred.notna() & y_true.notna()
            correct = (y_pred[mask] == y_true[mask]).astype(int)
            results.append(correct)
        
        common_units = results[0].index.intersection(results[1].index)
        a_correct = results[0].loc[common_units]
        b_correct = results[1].loc[common_units]
        
        # contingency table
        b = ((a_correct == 1) & (b_correct == 0)).sum()  # tuned richtig, base falsch
        c = ((a_correct == 0) & (b_correct == 1)).sum()  # tuned falsch, base richtig
        
        if b + c == 0:
            print(f"{rater:<8} {dim:<6} {'no discordant pairs':>25}")
            continue
        
        # mcnemar p-Wert
        table = np.array([
            [(( a_correct == 1) & (b_correct == 1)).sum(), b],
            [c, ((a_correct == 0) & (b_correct == 0)).sum()]
        ])
        result = mcnemar(table, exact=True)
        
        # necessary n: based on b/c ratio
        # kinda rule: for power 0.8 with alpha 0.05 mcnemar needs
        # n = (z_alpha/2 + z_beta)^2 / (p_b - p_c)^2 * (p_b + p_c)
        p_b = b / (b + c)
        p_c = c / (b + c)
        if p_b != p_c:
            effect = abs(p_b - p_c)
            n_needed = int(np.ceil((1.96 + 0.842)**2 * (p_b + p_c) / (p_b - p_c)**2))
        else:
            n_needed = np.inf
        
        print(f"{rater:<8} {dim:<6} {b:>5} {c:>5} {result.pvalue:>8.4f} {n_needed:>10}")

Rater    Dim        b     c        p   n_needed
---------------------------------------------
GPT      IE         6     5   1.0000        950
GPT      NS         6    13   0.1671         58
GPT      FT         5    14   0.0636         35
GPT      PJ         4     4   1.0000        inf
Opus     IE         7    13   0.2632         88
Opus     NS         3     8   0.2266         38
Opus     FT         2     8   0.1094         22
Opus     PJ         2    12   0.0129         16
Gemini   IE         5    12   0.1435         47
Gemini   NS         7    12   0.3593        114
Gemini   FT         1     9   0.0215         13
Gemini   PJ         6    11   0.3323         91


In [ ]:
# per rater

print(f"{'Dim':<6} {'b':>5} {'c':>5} {'p':>8} {'n_needed':>10}")
print("-" * 35)

for dim in dimensionen:
    all_tuned = []
    all_base = []
    
    for rater in ['GPT', 'Opus', 'Gemini']:
        results = []
        for model_df in [df, df_base]:
            rater_df = model_df[model_df['rater'] == rater].set_index('unit')
            common = rater_df.index.intersection(gt.index)
            y_pred = rater_df.loc[common, f'guess_{dim}'].dropna()
            y_true = gt.loc[y_pred.index, dim]
            mask = y_pred.notna() & y_true.notna()
            correct = (y_pred[mask] == y_true[mask]).astype(int)
            results.append(correct)
        
        common_units = results[0].index.intersection(results[1].index)
        all_tuned.append(results[0].loc[common_units])
        all_base.append(results[1].loc[common_units])
    
    # putting rater together
    a_correct = pd.concat(all_tuned)
    b_correct = pd.concat(all_base)
    
    b = ((a_correct == 1) & (b_correct == 0)).sum()
    c = ((a_correct == 0) & (b_correct == 1)).sum()
    
    if b + c == 0:
        print(f"{dim:<6} {'no discordant pairs':>30}")
        continue
    
    table = np.array([
        [((a_correct == 1) & (b_correct == 1)).sum(), b],
        [c, ((a_correct == 0) & (b_correct == 0)).sum()]
    ])
    result = mcnemar(table, exact=True)
    
    p_b = b / (b + c)
    p_c = c / (b + c)
    if p_b != p_c:
        n_needed = int(np.ceil((1.96 + 0.842)**2 * (p_b + p_c) / (p_b - p_c)**2))
    else:
        n_needed = np.inf
    
    print(f"{dim:<6} {b:>5} {c:>5} {result.pvalue:>8.4f} {n_needed:>10}")

Dim        b     c        p   n_needed
-----------------------------------
IE        18    30   0.1114        126
NS        16    33   0.0213         66
FT         8    31   0.0003         23
PJ        12    27   0.0237         54
